In [4]:
from __future__ import annotations

import argparse
from pathlib import Path

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(tf.__version__)

from .config import CHECKPOINTS, DATA_PROCESSED, TRAIN




2.21.0


ImportError: attempted relative import with no known parent package

*future* is used in python if you are using an older version but you want to borrow some of the features from future versions of python

*argparse* -> it is a built in python module that helps in craeting command line interfaces. It shows help messages and issues errors

*pathlib* -> makes the code readable when creating code that is used for searching path of a file it uses __/__

*config* -> in python it is an inbuilt feature to work with config files like .ini and .cfg

In [2]:
CACHE = DATA_PROCESSED / "uci_har"

#There is 9 channels in order:
#0,1,2 body_acc(x,y,z)
#3,4,5 body_gyro(x,y,z)
#6,7,8 total_acc(x,y,z)

#The training on total_acc + body_gyro only : needed by the MPU6050
#body_acc is the total_acc with gravity filtered
CHANNEL_IDX = [6, 7, 8, 3, 4, 5]  # ax, ay, az, gx, gy, gz

WINDOW = 128
N_CLASSES = 6
CLASS_NAMES = [
    "WALKING",
    "WALKING_UPSTAIRS",
    "WALKING_DOWNSTAIRS",
    "SITTING",
    "STANDING",
    "LAYING",
]


NameError: name 'DATA_PROCESSED' is not defined

**Line by line**

```python
CACHE = DATA_PROCESSED / "uci_har"
```
Path to the cached dataset folder (`data/processed/uci_har/`). `/` is `pathlib` path joining. `DATA_PROCESSED` comes from `config.py`.

```python
#There is 9 channels in order: ...
```
Comments describing the cached array layout. Each sample is `(128, 9)`: 128 time steps × 9 signals.
- 0-2: `body_acc` x/y/z (acceleration with gravity removed)
- 3-5: `body_gyro` x/y/z (rotation rate)
- 6-8: `total_acc` x/y/z (raw acceleration, gravity included)

```python
CHANNEL_IDX = [6, 7, 8, 3, 4, 5]  # ax, ay, az, gx, gy, gz
```
Which of the 9 channels the model uses: `total_acc` (6,7,8) and `body_gyro` (3,4,5). Later code does `X[:, :, CHANNEL_IDX]` to pick those 6 and reorder them as accel then gyro. That gives the 6 channels the model takes as input.

```python
WINDOW = 128
```
Samples per window: 128 at 50 Hz = 2.56 s. This sets the model's input length.

```python
N_CLASSES = 6
CLASS_NAMES = [...]
```
Number of output classes and their labels, in label-index order (class 0 = WALKING, and so on).

**Why gravity is filtered out (and why you *don't* want it)**

The comment is a bit misleading. The point of the code is that gravity is **not** filtered out of the channels the model uses.

- UCI HAR provides `body_acc`, which is `total_acc` with gravity removed. A Butterworth low-pass filter estimates the slow gravity component, and that is subtracted. The dataset creators did this for you.
- The MPU6050 only outputs **raw** acceleration, which is `total_acc`. It has no gravity-removed output.
- If the model trained on `body_acc`, the FPGA would need that same Butterworth filter in RTL before inference. Otherwise the live input wouldn't match the training data.
- So the code trains on `total_acc` plus the gyro. That matches what the sensor produces, and no filter is needed on-device.

Gravity also carries useful information. It tells you the sensor's orientation. That is how the model separates SITTING, STANDING and LAYING, which have almost no body motion. `body_acc` would be near zero for all three. Dropping it would hurt those classes, so `total_acc` is better for the model as well as simpler for the hardware.

One small fix: the comment says "total_acc + body_acc", but it should say "total_acc + body_gyro". The code itself is right.

In [ ]:
def load(split: str, cache: Path = CACHE):
    x_path, y_path = cache / f"X_{split}.npy", cache / f"Y_{split}.npy"
    return np.load(x_path)[:, :, CHANNEL_IDX], np.load(y_path)


This loads the saved dataset to be used.
the "def_load()" func is used for splitting it into train and test.
cache is the folder that holds all the data and it defaults to "CACHE"
the code for cache is made somwhere else.

x_path, y_path = cache / f"X_{split}.npy", cache / f"Y_{split}.npy"
this is for splitting the test and training dataset which are the sensor values which is going to be checked against the labels.

np.load(path) : this read the array which is of the format (num_windows,128,9)
[:, :, CHANNEL_IDX] means "all windows, all 128 time steps, but only channels 6,7,8,3,4,5 because 0,1,2 are part of the testing dataset so that part is excluded out. The result is (num_windows, 128, 6): accelerometer x/y/z, then gyro x/y/z.
np.load(y_path) reads the labels unchanged.
It returns both as a pair, so you call it as X, y = load("train").






In [ ]:
def build(filters = 16, hidden=16, dense=16, dropout=0.3, window=WINDOW, channels = len(CHANNEL_IDX), n_classes=N_CLASSES):
    model = keras.Sequential(name="cnnlastm")
    model.add(keras.Input(shape = (window, channels), name = "imu_windows"))
    model.add(layers.Conv1D(filters, 5, use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.ReLU())
    model.add(layers.Conv1D(filters, 5, use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.ReLU())
    model.add(layers.MaxPooling1D(2))
    model.add(layers.Dropout(dropout))
    model.add(layers.LSTM(hidden))
    model.add(layers.Dropout(dropout))
    model.add(layers.Dense(dense, activation="relu"))
    model.add(layers.Dense(n_classes, name="logits"))
    return model



In [ ]:
def train(model, epochs, out):
    x_train, y_train = load("train")
    x_test, y_test = load("test")
    
